# Panel estimators: event studies, estimates, and placebo SEs

This notebook compares the torch panel estimators on:

1. a synthetic latent-factor panel with selection on latent factors, where parallel trends fails; and
2. the California Proposition 99 smoking dataset from the synthdid reference repo.

It generates event-study plots and estimate/SE tables for DID, SC, SDID variants, DIFP variants, and matrix completion.

In [ ]:
from pathlib import Path
import csv, math, random
import torch
import matplotlib.pyplot as plt

from trex.panel import (
    SyntheticDID,
    did_estimate,
    matrix_completion_estimate,
    panel_estimates,
)

OUT = Path('nb/panel_estimators_event_study_outputs')
OUT.mkdir(parents=True, exist_ok=True)
METHODS = [
    'DID', 'Synthetic Control (SC)', 'Synthetic DID (SDID)', 'Time Weighted DID',
    'SDID (No Intercept)', 'SC with FEs (DIFP)', 'Matrix Completion',
    'SC (Regularized)', 'DIFP (Regularized)'
]


def make_synthetic_factor_panel(seed=123, n=100, n1=20, t=50, t0=40, rank=3, tau=-2.0):
    torch.manual_seed(seed)
    dtype = torch.float64
    n0 = n - n1
    F = torch.randn(n, rank, dtype=dtype)
    # Treatment selection on the first latent factor: high-factor units treated.
    order = torch.argsort(F[:, 0])
    control_idx = order[:n0]
    treated_idx = order[n0:]
    idx = torch.cat([control_idx, treated_idx])
    F = F[idx]
    time = torch.linspace(-1.0, 1.0, t, dtype=dtype)
    Lambda = torch.stack([
        2.0 * time,
        torch.sin(math.pi * time),
        (time + 0.25).pow(2),
    ], dim=1)[:, :rank]
    unit_fe = 0.8 * torch.randn(n, dtype=dtype)
    time_fe = 1.5 * time + 0.5 * torch.sin(2 * math.pi * (time + 1) / 2)
    # Selection on F[:,0] plus trending loading makes untreated treated units diverge pre/post.
    Y0 = F @ Lambda.T + unit_fe[:, None] + time_fe[None, :] + 0.25 * torch.randn(n, t, dtype=dtype)
    Y = Y0.clone()
    Y[n0:, t0:] += tau
    W = torch.zeros_like(Y, dtype=torch.bool)
    W[n0:, t0:] = True
    return Y, W, n0, t0, tau


def load_california_prop99(path=Path('../_refs/synthdid/data/california_prop99.csv')):
    rows = []
    with path.open(newline='') as f:
        reader = csv.DictReader(f, delimiter=';')
        for row in reader:
            rows.append((row['State'], int(row['Year']), float(row['PacksPerCapita']), int(row['treated'])))
    states_all = sorted({r[0] for r in rows})
    treated_states = sorted({r[0] for r in rows if r[3] == 1})
    states = [s for s in states_all if s not in treated_states] + treated_states
    years = sorted({r[1] for r in rows})
    by_key = {(s, y): (packs, treated) for s, y, packs, treated in rows}
    Y = torch.empty((len(states), len(years)), dtype=torch.float64)
    W = torch.empty_like(Y, dtype=torch.bool)
    for i, state in enumerate(states):
        for j, year in enumerate(years):
            Y[i, j], W[i, j] = by_key[(state, year)]
    N0 = int((~W.any(dim=1)).sum())
    T0 = int((~W.any(dim=0)).sum())
    return Y, W, N0, T0, states, years


def fit_weighted_method(Y, N0, T0, method, sdid_kwargs=None, mc_kwargs=None):
    sdid_kwargs = dict(sdid_kwargs or {})
    mc_kwargs = dict(mc_kwargs or {})
    N, T = Y.shape
    uniform_lambda = torch.full((T0,), 1.0 / T0, dtype=Y.dtype, device=Y.device)
    uniform_omega = torch.full((N0,), 1.0 / N0, dtype=Y.dtype, device=Y.device)
    reg_eta = ((N - N0) * (T - T0)) ** 0.25
    if method == 'DID':
        return {'estimate': did_estimate(Y, N0, T0), 'omega': uniform_omega, 'lambda': uniform_lambda, 'kind': 'weighted'}
    if method == 'Synthetic Control (SC)':
        r = SyntheticDID(eta_omega=1e-6, lambda_weights=torch.zeros(T0, dtype=Y.dtype), update_lambda=False, omega_intercept=False, **sdid_kwargs).fit(Y, N0, T0).result_
        return {'estimate': r.estimate, 'omega': r.omega, 'lambda': r.lambda_, 'kind': 'weighted'}
    if method == 'Synthetic DID (SDID)':
        r = SyntheticDID(**sdid_kwargs).fit(Y, N0, T0).result_
        return {'estimate': r.estimate, 'omega': r.omega, 'lambda': r.lambda_, 'kind': 'weighted'}
    if method == 'Time Weighted DID':
        r = SyntheticDID(omega=uniform_omega, update_omega=False, **sdid_kwargs).fit(Y, N0, T0).result_
        return {'estimate': r.estimate, 'omega': r.omega, 'lambda': r.lambda_, 'kind': 'weighted'}
    if method == 'SDID (No Intercept)':
        r = SyntheticDID(omega_intercept=False, **sdid_kwargs).fit(Y, N0, T0).result_
        return {'estimate': r.estimate, 'omega': r.omega, 'lambda': r.lambda_, 'kind': 'weighted'}
    if method == 'SC with FEs (DIFP)':
        r = SyntheticDID(lambda_weights=uniform_lambda, update_lambda=False, eta_omega=1e-6, **sdid_kwargs).fit(Y, N0, T0).result_
        return {'estimate': r.estimate, 'omega': r.omega, 'lambda': r.lambda_, 'kind': 'weighted'}
    if method == 'SC (Regularized)':
        r = SyntheticDID(eta_omega=reg_eta, lambda_weights=torch.zeros(T0, dtype=Y.dtype), update_lambda=False, omega_intercept=False, **sdid_kwargs).fit(Y, N0, T0).result_
        return {'estimate': r.estimate, 'omega': r.omega, 'lambda': r.lambda_, 'kind': 'weighted'}
    if method == 'DIFP (Regularized)':
        r = SyntheticDID(lambda_weights=uniform_lambda, update_lambda=False, **sdid_kwargs).fit(Y, N0, T0).result_
        return {'estimate': r.estimate, 'omega': r.omega, 'lambda': r.lambda_, 'kind': 'weighted'}
    if method == 'Matrix Completion':
        tau, model = matrix_completion_estimate(Y, N0, T0, **mc_kwargs)
        return {'estimate': tau, 'model': model, 'kind': 'mc'}
    raise ValueError(method)


def event_curve(Y, N0, T0, fit):
    N, T = Y.shape
    N1 = N - N0
    if fit['kind'] == 'weighted':
        omega = fit['omega']
        lam = fit['lambda']
        lambda_synth = torch.cat([lam, torch.zeros(T - T0, dtype=Y.dtype)])
        omega_synth = torch.cat([omega, torch.zeros(N1, dtype=Y.dtype)])
        omega_target = torch.cat([torch.zeros(N0, dtype=Y.dtype), torch.full((N1,), 1.0 / N1, dtype=Y.dtype)])
        offset = (omega_target - omega_synth) @ Y @ lambda_synth
        obs = omega_target @ Y
        syn = omega_synth @ Y + offset
        return obs - syn
    completed = fit['model'].predict()
    return Y[N0:, :].mean(dim=0) - completed[N0:, :].mean(dim=0)


def placebo_se(Y, N0, T0, method, reps=80, seed=0, sdid_kwargs=None, mc_kwargs=None):
    rng = random.Random(seed)
    N1 = Y.shape[0] - N0
    controls = list(range(N0))
    vals = []
    if N1 == 1 and len(controls) <= 80:
        draws = [[i] for i in controls]
    else:
        draws = [rng.sample(controls, N1) for _ in range(reps)]
    for treated in draws:
        treated = list(treated)
        pseudo_controls = [i for i in controls if i not in treated]
        idx = pseudo_controls + treated
        try:
            val = fit_weighted_method(Y[idx, :], len(pseudo_controls), T0, method, sdid_kwargs=sdid_kwargs, mc_kwargs=mc_kwargs)['estimate']
            if torch.isfinite(val):
                vals.append(float(val))
        except Exception:
            pass
    if len(vals) < 2:
        return float('nan')
    return float(torch.tensor(vals, dtype=torch.float64).std(unbiased=True))


def analyze_dataset(name, Y, N0, T0, x_labels=None, true_tau=None):
    sdid_kwargs = {'maxiter': 10_000, 'min_decrease': 1e-5, 'sparsify': True}
    mc_kwargs = {'lambda_fraction': 0.15, 'maxiter': 400, 'tol': 1e-7}
    fits = {m: fit_weighted_method(Y, N0, T0, m, sdid_kwargs=sdid_kwargs, mc_kwargs=mc_kwargs) for m in METHODS}
    rows = []
    for m, fit in fits.items():
        if m == 'Matrix Completion':
            se = placebo_se(Y, N0, T0, m, reps=12, seed=22, sdid_kwargs=sdid_kwargs, mc_kwargs={'lambda_fraction': 0.15, 'maxiter': 50, 'tol': 1e-5})
        else:
            se = placebo_se(Y, N0, T0, m, reps=25, seed=22, sdid_kwargs={**sdid_kwargs, 'maxiter': 3000}, mc_kwargs={'lambda_fraction': 0.15, 'maxiter': 50, 'tol': 1e-5})
        rows.append((m, float(fit['estimate']), se))

    fig, ax = plt.subplots(figsize=(11, 6))
    xs = list(range(Y.shape[1])) if x_labels is None else x_labels
    for m in METHODS:
        curve = event_curve(Y, N0, T0, fits[m]).detach().cpu().numpy()
        lw = 2.4 if m in ['Synthetic DID (SDID)', 'Matrix Completion', 'DID'] else 1.2
        alpha = 0.95 if m in ['Synthetic DID (SDID)', 'Matrix Completion', 'DID'] else 0.65
        ax.plot(xs, curve, label=m, linewidth=lw, alpha=alpha)
    x_treat = xs[T0]
    ax.axvline(x_treat, color='black', linestyle='--', linewidth=1)
    ax.axhline(0, color='gray', linewidth=0.8)
    if true_tau is not None:
        ax.axhline(true_tau, color='black', linestyle=':', linewidth=1.2, label=f'true ATT = {true_tau:g}')
    ax.set_title(f'{name}: event-study curves')
    ax.set_xlabel('time')
    ax.set_ylabel('estimated effect curve')
    ax.legend(ncol=2, fontsize=8)
    fig.tight_layout()
    fig_path = OUT / f'{name.lower().replace(" ", "_")}_event_study.png'
    fig.savefig(fig_path, dpi=180)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 4.8))
    labels = [r[0] for r in rows]
    est = [r[1] for r in rows]
    ses = [r[2] for r in rows]
    y_pos = list(range(len(rows)))
    ax.errorbar(est, y_pos, xerr=[1.96 * s if math.isfinite(s) else 0 for s in ses], fmt='o', capsize=3)
    if true_tau is not None:
        ax.axvline(true_tau, color='black', linestyle=':', label='true ATT')
    ax.axvline(0, color='gray', linewidth=0.8)
    ax.set_yticks(y_pos, labels)
    ax.invert_yaxis()
    ax.set_xlabel('ATT estimate with placebo 95% interval')
    ax.set_title(f'{name}: estimates and placebo SEs')
    fig.tight_layout()
    est_path = OUT / f'{name.lower().replace(" ", "_")}_estimates.png'
    fig.savefig(est_path, dpi=180)
    plt.close(fig)

    tex = ['| Method | Estimate | Placebo SE |', '|---|---:|---:|']
    for m, e, se in rows:
        tex.append(f'| {m} | {e:.3f} | {se:.3f} |')
    table_path = OUT / f'{name.lower().replace(" ", "_")}_estimates.md'
    table_path.write_text('\n'.join(tex) + '\n')
    return rows, fig_path, est_path, table_path


def run_all():
    Y_syn, W_syn, N0_syn, T0_syn, tau = make_synthetic_factor_panel()
    syn_rows, syn_fig, syn_est, syn_table = analyze_dataset('Synthetic factor', Y_syn, N0_syn, T0_syn, true_tau=tau)
    Y_ca, W_ca, N0_ca, T0_ca, states, years = load_california_prop99()
    ca_rows, ca_fig, ca_est, ca_table = analyze_dataset('California smoking', Y_ca, N0_ca, T0_ca, x_labels=years)
    return {
        'synthetic': (syn_rows, syn_fig, syn_est, syn_table),
        'california': (ca_rows, ca_fig, ca_est, ca_table),
    }

if __name__ == '__main__':
    out = run_all()
    for k, (_, fig, est, table) in out.items():
        print(k, fig, est, table)
        print(table.read_text())


Generated outputs are written under `nb/panel_estimators_event_study_outputs/`.
